# Datamart preflight checks (Feb 2026)

Упрощенная версия:
- линейный пайплайн по секциям;
- fallback `r2.id` только при пустом `agr_id`;
- детерминированная дедупликация и QC до сборки витрины.

In [ ]:
import re
from decimal import Decimal, InvalidOperation
from pathlib import Path

import pandas as pd
from rail_connectors.connection import connect

In [ ]:
report_month = '2026-02-01'
month_start = '2026-02-01'
month_end = '2026-02-28'

excel_path = '/home/jovyan/documents/Equaring/Data/02_Февраль_2026.xlsx'
excel_inn_col = 'ИНН'
excel_header = 0

mem_limit = '8g'

In [ ]:
def normalize_numeric_str(v):
    if pd.isna(v):
        return None
    s = str(v).strip().replace(' ', '').replace('\xa0', '')
    if s == '' or s.lower() in {'nan', 'none', 'null'}:
        return None
    s = s.replace(',', '.')
    if re.search(r'[eE][+-]?\d+$', s):
        try:
            s = format(Decimal(s), 'f')
        except InvalidOperation:
            return None
    s = re.sub(r'\.0+$', '', s)
    s = re.sub(r'\D', '', s)
    if not s:
        return None
    if len(s) == 9:
        s = '0' + s
    elif len(s) == 11:
        s = '0' + s
    return s


def normalize_contract(v):
    if pd.isna(v):
        return None
    s = str(v).strip().lower()
    s = re.sub(r'\s+', '', s)
    return s if s else None


def to_sql_in_list(values):
    return ', '.join(["'" + str(v).replace("'", "''") + "'" for v in values])


def chunk_list(values, chunk_size):
    for i in range(0, len(values), chunk_size):
        yield values[i:i + chunk_size]

In [ ]:
imp = connect(
    to='IMPALA',
    extra_options={'db': 'sandbox_ai'},
    driver_args={'tez.queue.name': 'ai'},
    kerberos={
        'keytab_path': '/home/jovyan/test_requests/tech.keytab',
        'use_credentials': True,
        'update_keytab': True,
    },
    user_params={'user_name': 'Shestopalov-VYur'}
)
imp._init_connection()

In [ ]:
excel_raw = pd.read_excel(excel_path, header=excel_header)
excel_raw.columns = [str(c).strip() for c in excel_raw.columns]

if excel_inn_col not in excel_raw.columns:
    raise ValueError(f'Missing Excel INN column: {excel_inn_col}')

excel_df = excel_raw[[excel_inn_col]].copy()
excel_df.columns = ['inn_raw']
excel_df['inn'] = excel_df['inn_raw'].apply(normalize_numeric_str)
excel_df = excel_df[(excel_df['inn'].notna()) & (excel_df['inn'] != '')].copy()
excel_df = excel_df[['inn']].drop_duplicates().sort_values('inn').reset_index(drop=True)

excel_inn_set = set(excel_df['inn'].tolist())
len(excel_inn_set)

In [ ]:
sql_agr = f"""
select distinct
  regexp_replace(trim(cast(c.c_inn as string)), '[^0-9]', '') as inn,
  cast(a.abs_agr_id as string) as agr_id,
  cast(a.c_agr_number as string) as agreement_contract_number
from ods_alpha.scd1_agreements a
join ods_alpha.scd1_companies c
  on c.n_cmp = a.n_cmp_client
where upper(trim(cast(a.acq_class as string))) = 'SA'
  and cast(a.d_valid_from as date) <= cast('{month_end}' as date)
  and (a.d_valid_to is null or cast(a.d_valid_to as date) >= cast('{month_start}' as date))
  and coalesce(a.ods_deleted_flg, '0') <> '1'
  and coalesce(c.ods_deleted_flg, '0') <> '1'
  and c.c_inn is not null
"""

with imp:
    imp.execute(f"set MEM_LIMIT={mem_limit}")
    agr_df = imp.fetch(sql_agr)

agr_df['inn'] = agr_df['inn'].apply(normalize_numeric_str)
agr_df['agr_id'] = agr_df['agr_id'].astype(str)
agr_df.loc[agr_df['agr_id'].isin(['None', 'nan', '']), 'agr_id'] = None
agr_df['agreement_contract_number'] = agr_df['agreement_contract_number'].apply(normalize_contract)
agr_df = agr_df[(agr_df['inn'].notna()) & (agr_df['inn'] != '')].drop_duplicates().reset_index(drop=True)
agr_df['row_uid'] = agr_df.index.astype(str)

agr_id_list = sorted(agr_df['agr_id'].dropna().unique().tolist())
inn_list = sorted(agr_df['inn'].dropna().unique().tolist())


def build_sql_r2_best_by_ids(in_list):
    return f"""
    with m as (
      select
        cast(id as string) as r2_id,
        cast(c_cl_org as string) as cft_id,
        cast(c_code_in_pr as string) as c_code_in_pr
      from ods.scd1_z_r2_ip_merchants
      where cast(id as string) in ({in_list})
    ),
    d as (
      select
        cast(id as string) as dog_id,
        cast(c_parent_id as string) as dog_parent_id,
        cast(c_code_in_pr as string) as dog_code_in_pr,
        cast(c_num as string) as dog_contract_number
      from ods.scd1_z_r2_ip_merch_dog
    ),
    candidate as (
      select m.r2_id, m.cft_id, m.c_code_in_pr, d.dog_contract_number, 'join_by_parent_id' as join_type, 1 as rank, d.dog_id
      from m left join d on d.dog_parent_id = m.r2_id
      union all
      select m.r2_id, m.cft_id, m.c_code_in_pr, d.dog_contract_number, 'join_by_id' as join_type, 2 as rank, d.dog_id
      from m left join d on d.dog_id = m.r2_id
      union all
      select m.r2_id, m.cft_id, m.c_code_in_pr, d.dog_contract_number, 'join_by_code_in_pr' as join_type, 3 as rank, d.dog_id
      from m left join d on d.dog_code_in_pr is not null and m.c_code_in_pr is not null and d.dog_code_in_pr = m.c_code_in_pr
      union all
      select m.r2_id, m.cft_id, m.c_code_in_pr, cast(null as string) as dog_contract_number, 'no_match' as join_type, 9 as rank, cast(null as string) as dog_id
      from m
    ),
    ranked as (
      select *, row_number() over (
        partition by r2_id
        order by rank, case when dog_contract_number is null then 1 else 0 end, dog_id
      ) as rn
      from candidate
    )
    select r2_id, cft_id, c_code_in_pr, dog_contract_number, join_type
    from ranked
    where rn = 1
    """


def build_sql_inn_to_cdi(in_list):
    return f"""
    with ocrm as (
      select
        regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') as inn,
        cast(se.row_id as string) as row_id,
        row_number() over (
          partition by regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '')
          order by cast(se.last_upd as timestamp) desc,
                   cast(se.created as timestamp) desc,
                   cast(se.row_id as string) desc
        ) as rn
      from ocrm_ul.s_org_ext se
      where se.x_removed_flg = 'N'
        and se.x_duplicate_flg = 'N'
        and regexp_replace(trim(cast(se.x_inn as string)), '[^0-9]', '') in ({in_list})
    )
    select
      o.inn,
      cast(e.party_id as string) as cdi_id
    from ocrm o
    left join cdiul.ext_id_org e
      on cast(e.cmo_ext_party_source_id as string) = o.row_id
     and upper(e.cmo_ext_source_system) like 'OCRM%'
    where o.rn = 1
    """


def build_sql_cdi_to_cft(in_list):
    return f"""
    select distinct
      cast(party_id as string) as cdi_id,
      cast(cmo_ext_party_source_id as string) as cft_id
    from cdiul.ext_id_org
    where upper(cmo_ext_source_system) like 'CFT%'
      and cast(party_id as string) in ({in_list})
    """


def fetch_chunked(builder, values, chunk_size):
    dfs = []
    for part in chunk_list(values, chunk_size):
        in_list = to_sql_in_list(part)
        with imp:
            imp.execute(f"set MEM_LIMIT={mem_limit}")
            d = imp.fetch(builder(in_list))
        if len(d):
            dfs.append(d)
    return pd.concat(dfs, ignore_index=True) if len(dfs) else pd.DataFrame()


r2_best_df = fetch_chunked(build_sql_r2_best_by_ids, agr_id_list, 1200) if len(agr_id_list) else pd.DataFrame(columns=['r2_id', 'cft_id', 'c_code_in_pr', 'dog_contract_number', 'join_type'])
r2_best_df['dog_contract_number'] = r2_best_df['dog_contract_number'].apply(normalize_contract) if len(r2_best_df) else r2_best_df.get('dog_contract_number')
r2_best_df = r2_best_df.drop_duplicates('r2_id') if len(r2_best_df) else r2_best_df

inn_to_cdi_df = fetch_chunked(build_sql_inn_to_cdi, inn_list, 800) if len(inn_list) else pd.DataFrame(columns=['inn', 'cdi_id'])
if len(inn_to_cdi_df):
    inn_to_cdi_df['inn'] = inn_to_cdi_df['inn'].apply(normalize_numeric_str)
    inn_to_cdi_df['cdi_id'] = inn_to_cdi_df['cdi_id'].astype(str)
    inn_to_cdi_df.loc[inn_to_cdi_df['cdi_id'].isin(['None', 'nan', '']), 'cdi_id'] = None
    inn_to_cdi_df = inn_to_cdi_df.drop_duplicates()

cdi_list = sorted(inn_to_cdi_df['cdi_id'].dropna().unique().tolist()) if len(inn_to_cdi_df) else []
cdi_to_cft_df = fetch_chunked(build_sql_cdi_to_cft, cdi_list, 1500) if len(cdi_list) else pd.DataFrame(columns=['cdi_id', 'cft_id'])
if len(cdi_to_cft_df):
    cdi_to_cft_df['cdi_id'] = cdi_to_cft_df['cdi_id'].astype(str)
    cdi_to_cft_df['cft_id'] = cdi_to_cft_df['cft_id'].astype(str)
    cdi_to_cft_df.loc[cdi_to_cft_df['cft_id'].isin(['None', 'nan', '']), 'cft_id'] = None
    cdi_to_cft_df = cdi_to_cft_df.drop_duplicates()

cdi_to_cft_one = cdi_to_cft_df.sort_values('cft_id').drop_duplicates('cdi_id') if len(cdi_to_cft_df) else cdi_to_cft_df
inn_cft_df = inn_to_cdi_df.merge(cdi_to_cft_one, on='cdi_id', how='left').drop_duplicates() if len(inn_to_cdi_df) else pd.DataFrame(columns=['inn', 'cdi_id', 'cft_id'])

# Tier1: строки с исходным agr_id
base_tier1 = agr_df[agr_df['agr_id'].notna()].copy()
base_tier1 = base_tier1.merge(
    r2_best_df[['r2_id', 'cft_id', 'dog_contract_number', 'join_type']],
    left_on='agr_id',
    right_on='r2_id',
    how='left'
)
base_tier1 = base_tier1.merge(inn_to_cdi_df[['inn', 'cdi_id']], on='inn', how='left')
base_tier1['fallback_r2_id'] = None
base_tier1['fallback_join_type'] = None

# Tier2: fallback только для строк без agr_id
base_tier2 = agr_df[agr_df['agr_id'].isna()].copy()
if len(base_tier2):
    t2 = base_tier2.merge(inn_cft_df[['inn', 'cdi_id', 'cft_id']], on='inn', how='left')
    cft_list = sorted(t2['cft_id'].dropna().unique().tolist())

    def build_sql_r2_best_by_cft(in_list):
        return f"""
        with m as (
          select
            cast(id as string) as r2_id,
            cast(c_cl_org as string) as cft_id,
            cast(c_code_in_pr as string) as c_code_in_pr
          from ods.scd1_z_r2_ip_merchants
          where cast(c_cl_org as string) in ({in_list})
        ),
        d as (
          select
            cast(id as string) as dog_id,
            cast(c_parent_id as string) as dog_parent_id,
            cast(c_code_in_pr as string) as dog_code_in_pr,
            cast(c_num as string) as dog_contract_number
          from ods.scd1_z_r2_ip_merch_dog
        ),
        candidate as (
          select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_parent_id' as join_type, 1 as rank, d.dog_id
          from m left join d on d.dog_parent_id = m.r2_id
          union all
          select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_id' as join_type, 2 as rank, d.dog_id
          from m left join d on d.dog_id = m.r2_id
          union all
          select m.r2_id, m.cft_id, d.dog_contract_number, 'join_by_code_in_pr' as join_type, 3 as rank, d.dog_id
          from m left join d on d.dog_code_in_pr is not null and m.c_code_in_pr is not null and d.dog_code_in_pr = m.c_code_in_pr
          union all
          select m.r2_id, m.cft_id, cast(null as string) as dog_contract_number, 'no_match' as join_type, 9 as rank, cast(null as string) as dog_id
          from m
        ),
        ranked as (
          select *, row_number() over (
            partition by r2_id
            order by rank, case when dog_contract_number is null then 1 else 0 end, dog_id
          ) as rn
          from candidate
        )
        select r2_id, cft_id, dog_contract_number, join_type
        from ranked
        where rn = 1
        """

    r2_cft_df = fetch_chunked(build_sql_r2_best_by_cft, cft_list, 1200) if len(cft_list) else pd.DataFrame(columns=['r2_id', 'cft_id', 'dog_contract_number', 'join_type'])
    if len(r2_cft_df):
        r2_cft_df['dog_contract_number'] = r2_cft_df['dog_contract_number'].apply(normalize_contract)

    t2 = t2.merge(r2_cft_df[['r2_id', 'cft_id', 'dog_contract_number', 'join_type']], on='cft_id', how='left')
    t2['match_priority'] = 3
    t2.loc[
        t2['agreement_contract_number'].notna() &
        t2['dog_contract_number'].notna() &
        (t2['agreement_contract_number'] == t2['dog_contract_number']),
        'match_priority'
    ] = 1
    t2.loc[
        t2['agreement_contract_number'].isna() | t2['dog_contract_number'].isna(),
        'match_priority'
    ] = t2.loc[
        t2['agreement_contract_number'].isna() | t2['dog_contract_number'].isna(),
        'match_priority'
    ].clip(upper=2)
    t2 = t2.sort_values(['row_uid', 'match_priority', 'r2_id'])

    t2_best = t2.groupby('row_uid', as_index=False).first()
    t2_best['fallback_r2_id'] = t2_best['r2_id']
    t2_best['fallback_join_type'] = t2_best['join_type']

    base_tier2 = base_tier2.merge(
        t2_best[['row_uid', 'cdi_id', 'cft_id', 'dog_contract_number', 'fallback_r2_id', 'fallback_join_type']],
        on='row_uid',
        how='left'
    )
    base_tier2['r2_id'] = base_tier2['fallback_r2_id']
    base_tier2['join_type'] = base_tier2['fallback_join_type']
else:
    base_tier2['cdi_id'] = None
    base_tier2['cft_id'] = None
    base_tier2['dog_contract_number'] = None
    base_tier2['fallback_r2_id'] = None
    base_tier2['fallback_join_type'] = None
    base_tier2['r2_id'] = None
    base_tier2['join_type'] = None

base_df = pd.concat([base_tier1, base_tier2], ignore_index=True, sort=False)
base_df['dog_contract_number'] = base_df['dog_contract_number'].apply(normalize_contract)
base_df['contract_number_final'] = base_df['dog_contract_number'].where(base_df['dog_contract_number'].notna(), base_df['agreement_contract_number'])
base_df['agr_id_final'] = base_df['agr_id'].where(base_df['agr_id'].notna(), base_df['fallback_r2_id'])
base_df['key_source'] = base_df['agr_id'].apply(lambda x: 'agr_id' if pd.notna(x) and str(x).strip() != '' else 'r2_fallback')
base_df['join_type'] = base_df['join_type'].fillna('no_match')

base_df = base_df[[
    'row_uid', 'inn', 'agr_id', 'r2_id', 'fallback_r2_id', 'key_source', 'join_type',
    'agreement_contract_number', 'dog_contract_number', 'contract_number_final',
    'cdi_id', 'cft_id', 'agr_id_final'
]].drop_duplicates().reset_index(drop=True)

stage_counts_df = pd.DataFrame([
    {'stage': '01_stage_agr', 'row_cnt': len(agr_df), 'inn_cnt': agr_df['inn'].nunique(), 'agr_cnt': agr_df['agr_id'].nunique(dropna=True)},
    {'stage': '02_tier1_after_r2', 'row_cnt': len(base_tier1), 'inn_cnt': base_tier1['inn'].nunique(), 'agr_cnt': base_tier1['agr_id'].nunique(dropna=True)},
    {'stage': '03_tier2_candidates', 'row_cnt': len(base_tier2), 'inn_cnt': base_tier2['inn'].nunique(), 'agr_cnt': base_tier2['agr_id'].nunique(dropna=True)},
    {'stage': '04_final_base_df', 'row_cnt': len(base_df), 'inn_cnt': base_df['inn'].nunique(), 'agr_cnt': base_df['agr_id_final'].nunique(dropna=True)},
])

len(base_df)

In [ ]:
lake_inn_set = set(base_df['inn'].dropna().tolist())
in_both_set = excel_inn_set & lake_inn_set
only_excel_set = excel_inn_set - lake_inn_set
only_lake_set = lake_inn_set - excel_inn_set

fallback_df = base_df[['inn', 'join_type', 'agreement_contract_number', 'dog_contract_number']].drop_duplicates()
join_stat = fallback_df.groupby('join_type', as_index=False).agg(inn_cnt=('inn', 'nunique')).sort_values('inn_cnt', ascending=False)

excel_eval = pd.DataFrame({'inn': sorted(excel_inn_set)}).merge(
    base_df[['inn', 'cdi_id', 'cft_id', 'contract_number_final', 'agr_id_final']].drop_duplicates(),
    on='inn',
    how='left'
)

excel_eval['has_cdi'] = excel_eval['cdi_id'].notna() & (excel_eval['cdi_id'].astype(str) != '')
excel_eval['has_cft'] = excel_eval['cft_id'].notna() & (excel_eval['cft_id'].astype(str) != '')
excel_eval['has_contract_final'] = excel_eval['contract_number_final'].notna() & (excel_eval['contract_number_final'].astype(str) != '')
excel_eval['has_agr_id_final'] = excel_eval['agr_id_final'].notna() & (excel_eval['agr_id_final'].astype(str) != '')

excel_eval_agg = (
    excel_eval.groupby('inn', as_index=False)
    .agg(
        has_cdi=('has_cdi', 'max'),
        has_cft=('has_cft', 'max'),
        has_contract_final=('has_contract_final', 'max'),
        has_agr_id_final=('has_agr_id_final', 'max')
    )
)

key_df = base_df[['inn', 'contract_number_final', 'agr_id_final', 'join_type', 'agreement_contract_number', 'dog_contract_number']].copy()
key_df = key_df[key_df['inn'].notna() & key_df['contract_number_final'].notna()].copy()
key_dup_df = key_df.groupby(['inn', 'contract_number_final'], as_index=False).agg(
    rows=('agr_id_final', 'size'),
    agr_id_final_nunique=('agr_id_final', 'nunique')
)
key_dup_df = key_dup_df[(key_dup_df['rows'] > 1) | (key_dup_df['agr_id_final_nunique'] > 1)].sort_values(['rows', 'agr_id_final_nunique'], ascending=False)

conflict_df = key_df.groupby(['inn', 'contract_number_final'], as_index=False).agg(agr_id_final_nunique=('agr_id_final', 'nunique'))
conflict_df = conflict_df[conflict_df['agr_id_final_nunique'] > 1].sort_values('agr_id_final_nunique', ascending=False)

dup_detail_df = base_df.merge(
    key_dup_df[['inn', 'contract_number_final']],
    on=['inn', 'contract_number_final'],
    how='inner'
).sort_values(['inn', 'contract_number_final', 'agr_id_final', 'join_type'])

conflict_detail_df = base_df.merge(
    conflict_df[['inn', 'contract_number_final']],
    on=['inn', 'contract_number_final'],
    how='inner'
).sort_values(['inn', 'contract_number_final', 'agr_id_final', 'join_type'])

key_source_stat = base_df.groupby('key_source', as_index=False).agg(rows=('inn', 'size'), inn_cnt=('inn', 'nunique'))
rows_total = len(base_df)
rows_fallback = int(base_df['key_source'].eq('r2_fallback').sum())
rows_primary = int(base_df['key_source'].eq('agr_id').sum())

summary_df = pd.DataFrame([
    {'metric': 'excel_unique_inn', 'value': len(excel_inn_set)},
    {'metric': 'lake_unique_inn', 'value': len(lake_inn_set)},
    {'metric': 'inn_in_both', 'value': len(in_both_set)},
    {'metric': 'inn_only_in_excel', 'value': len(only_excel_set)},
    {'metric': 'inn_only_in_lake', 'value': len(only_lake_set)},
    {'metric': 'excel_inn_with_cdi', 'value': int(excel_eval_agg['has_cdi'].sum())},
    {'metric': 'excel_inn_with_cft', 'value': int(excel_eval_agg['has_cft'].sum())},
    {'metric': 'excel_inn_with_contract_final', 'value': int(excel_eval_agg['has_contract_final'].sum())},
    {'metric': 'excel_inn_with_agr_id_final', 'value': int(excel_eval_agg['has_agr_id_final'].sum())},
    {'metric': 'rows_with_primary_agr_id', 'value': rows_primary},
    {'metric': 'rows_with_r2_fallback', 'value': rows_fallback},
    {'metric': 'rows_with_r2_fallback_pct', 'value': round(100.0 * rows_fallback / rows_total, 2) if rows_total else 0.0},
    {'metric': 'duplicate_key_cnt_inn_contract_final', 'value': len(key_dup_df)},
    {'metric': 'conflict_agr_id_final_cnt', 'value': len(conflict_df)},
])

summary_df

In [ ]:
examples_only_excel_df = pd.DataFrame({'inn': sorted(list(only_excel_set))[:5]})
examples_only_lake_df = pd.DataFrame({'inn': sorted(list(only_lake_set))[:5]})
examples_missing_contract_df = excel_eval[~excel_eval['has_contract_final']][['inn']].drop_duplicates().sort_values('inn').head(5).reset_index(drop=True)
examples_missing_agr_id_final_df = excel_eval[~excel_eval['has_agr_id_final']][['inn']].drop_duplicates().sort_values('inn').head(5).reset_index(drop=True)

examples_only_excel_df, examples_only_lake_df, examples_missing_contract_df, examples_missing_agr_id_final_df

In [ ]:
coverage_contract_pct = round(100.0 * int(excel_eval_agg['has_contract_final'].sum()) / len(excel_eval_agg), 2) if len(excel_eval_agg) else 0.0

fallback_df['is_no_match_hard'] = (
    (fallback_df['join_type'] == 'no_match') &
    (fallback_df['agreement_contract_number'].isna()) &
    (fallback_df['dog_contract_number'].isna())
)
no_match_hard_pct = round(100.0 * fallback_df['is_no_match_hard'].mean(), 2) if len(fallback_df) else 0.0

dup_cnt = len(key_dup_df)

qc_gate_df = pd.DataFrame([
    {'check': 'contract_final_coverage_ge_95pct', 'value': coverage_contract_pct, 'threshold': 95.0, 'pass': coverage_contract_pct >= 95.0},
    {'check': 'no_match_hard_le_5pct', 'value': no_match_hard_pct, 'threshold': 5.0, 'pass': no_match_hard_pct <= 5.0},
    {'check': 'duplicate_inn_contract_eq_0', 'value': dup_cnt, 'threshold': 0, 'pass': dup_cnt == 0},
])

qc_gate_df

In [ ]:
examples_df = pd.concat([
    examples_only_excel_df.assign(example_type='only_excel'),
    examples_only_lake_df.assign(example_type='only_lake'),
    examples_missing_contract_df.assign(example_type='missing_contract_final'),
    examples_missing_agr_id_final_df.assign(example_type='missing_agr_id_final'),
], ignore_index=True)

summary_df, key_source_stat, join_stat, qc_gate_df, stage_counts_df, key_dup_df.head(50), conflict_df.head(50), dup_detail_df.head(50), conflict_detail_df.head(50), examples_df

## Stage counts (from pipeline)

In [ ]:
stage_counts_df

In [ ]:
# 3) Compact diagnostic output
root_cause_kpi_df = pd.DataFrame([
    {
        'metric': 'missing_agr_id_final_rows',
        'value': int(base_df['agr_id_final'].isna().sum())
    },
    {
        'metric': 'missing_agr_id_final_inn',
        'value': int(base_df.loc[base_df['agr_id_final'].isna(), 'inn'].nunique())
    },
    {
        'metric': 'only_excel_inn_cnt',
        'value': len(only_excel_set)
    },
])

root_cause_kpi_df, src_presence_stat_df, root_cause_stat_df, root_cause_examples_df.head(30)